# Sudoku solver

A sudoku solver built on backtracking search. It fills the most constrained cell first (the cell with the fewest legal digits) and detects puzzles that are invalid or have no solution, returning a grid of `-1` for those.

Year 1 coursework, January 2024.

In [1]:
import numpy as np

# Load sudokus
sudoku = np.load("data/very_easy_puzzle.npy")
print("very_easy_puzzle.npy has been loaded into the variable sudoku")
print(f"sudoku.shape: {sudoku.shape}, sudoku[0].shape: {sudoku[0].shape}, sudoku.dtype: {sudoku.dtype}")

# Load solutions for demonstration
solutions = np.load("data/very_easy_solution.npy")
print()

# Print the first 9x9 sudoku...
print("First sudoku:")
print(sudoku[0], "\n")

# ...and its solution
print("Solution of first sudoku:")
print(solutions[0])

very_easy_puzzle.npy has been loaded into the variable sudoku
sudoku.shape: (15, 9, 9), sudoku[0].shape: (9, 9), sudoku.dtype: int8

First sudoku:
[[1 0 4 3 8 2 9 5 6]
 [2 0 5 4 6 7 1 3 8]
 [3 8 6 9 5 1 4 0 2]
 [4 6 1 5 2 3 8 9 7]
 [7 3 8 1 4 9 6 2 5]
 [9 5 2 8 7 6 3 1 4]
 [5 2 9 6 3 4 7 8 1]
 [6 0 7 2 9 8 5 4 3]
 [8 4 3 0 1 5 2 6 9]] 

Solution of first sudoku:
[[1 7 4 3 8 2 9 5 6]
 [2 9 5 4 6 7 1 3 8]
 [3 8 6 9 5 1 4 7 2]
 [4 6 1 5 2 3 8 9 7]
 [7 3 8 1 4 9 6 2 5]
 [9 5 2 8 7 6 3 1 4]
 [5 2 9 6 3 4 7 8 1]
 [6 1 7 2 9 8 5 4 3]
 [8 4 3 7 1 5 2 6 9]]


## The solver

1. `check_input` rejects grids where a digit repeats in a row, column or 3×3 box.
2. `fill_next_cell` picks the empty cell with the fewest valid options.
3. `solve` tries each valid digit in that cell, recurses, and undoes the move if it leads to a dead end.

In [2]:
def sudoku_solver(sudoku):
    """
    Solves a Sudoku puzzle and returns its unique solution.

    Input
        sudoku : 9x9 numpy array
            Empty cells are designated by 0.

    Output
        9x9 numpy array of integers
            It contains the solution, if there is one. If there is no solution, all array entries should be -1.
    """

    def check_input(board): #checks if the unsolved sudoku has valid initial state
        rows, cols, blocks = set(), set(), set()
        for i in range(9):
            for j in range(9):
                if board[i][j] != 0:
                    row_key = (i, board[i][j])
                    col_key = (j, board[i][j])
                    block_key = (i // 3, j // 3, board[i][j])
                    if row_key in rows or col_key in cols or block_key in blocks:
                        return False
                    rows.add(row_key)
                    cols.add(col_key)
                    blocks.add(block_key)
        return True

    def fill_next_cell(board): #finds the next cell to fill in 
        best_cell, min_options = None, 10
        for i in range(9):
            for j in range(9):
                if board[i][j] == 0:
                    options = sum(valid(board, i, j, num) for num in range(1, 10))
                    if options < min_options:
                        best_cell, min_options = (i, j), options
        return best_cell

    def valid(board, row, col, num): #checks if it is valid to put a number on the board
        if num in board[row]:  # checks row
            return False
        if num in [board[i][col] for i in range(9)]:  # checks column
            return False
        start_row, start_col = 3 * (row // 3), 3 * (col // 3)
        for i in range(3):
            for j in range(3):
                if board[i + start_row][j + start_col] == num:
                    return False
        return True

    def solve(board): #solves sudoku
        cell = fill_next_cell(board)
        if not cell:
            return True
        row, col = cell
        for num in range(1, 10):
            if valid(board, row, col, num):
                board[row][col] = num
                if solve(board):
                    return True
                board[row][col] = 0
        return False

    if not check_input(sudoku):
        return np.full((9, 9), -1)

    solved_sudoku = sudoku.copy()
    if solve(solved_sudoku):
        return solved_sudoku
    else:
        return np.full((9, 9), -1)
    
    



## Results

Tested on 60 puzzles across four difficulty levels. From "easy" upwards, a third of each set is invalid or unsolvable.

In [3]:
import time

for difficulty in ["very_easy", "easy", "medium", "hard"]:
    puzzles = np.load(f"data/{difficulty}_puzzle.npy")
    solutions = np.load(f"data/{difficulty}_solution.npy")
    correct, times = 0, []
    for puzzle, solution in zip(puzzles, solutions):
        start = time.perf_counter()
        answer = sudoku_solver(puzzle.copy())
        times.append(time.perf_counter() - start)
        correct += np.array_equal(answer, solution)
    print(f"{difficulty:<10} {correct}/{len(puzzles)} correct   average {np.mean(times):.3f} s   slowest {max(times):.2f} s")

very_easy  15/15 correct   average 0.001 s   slowest 0.00 s
easy       15/15 correct   average 0.001 s   slowest 0.00 s
medium     15/15 correct   average 0.006 s   slowest 0.01 s


hard       15/15 correct   average 3.579 s   slowest 13.91 s


Every puzzle is solved correctly, but hard puzzles take seconds. Each step rescans the whole board to count every empty cell's options. Tracking the remaining candidates for each row, column and box incrementally (constraint propagation) would be the next improvement.